**Navigation** : [Index](README.md) | [<< Précédent](22_TensorSharp_DotNet_Inference.ipynb) | [Suivant >>](../SemanticKernel/README.md)

# 23. LLamaSharp : bake-off binding .NET de llama.cpp

**Durée estimée** : 50 minutes
**Prérequis** : C# asynchrone, base de llama.cpp, AppLocker Windows
**Matériel du run de référence** : GPU NVIDIA RTX 3080 Ti 16 Go ; modèle GGUF Qwen3-4B Q4_K_M (~2.5 GB)

## Objectifs d'apprentissage

1. Brancher un binding .NET de `llama.cpp` (LLamaSharp 0.27.0) sur le même GPU que la Phase 1 (TensorSharp, PR #12645).
2. Charger un GGUF et mesurer `tok/s` (débit), VRAM, et taux de jetons `<pad>` — sans recopier un benchmark tiers.
3. Comparer à TensorSharp sur les mêmes invites et la même quantification Q4, et statuer sur l'utilité des deux moteurs pour le curriculum.
4. Documenter un blocage d'environnement reproductible (AppLocker Windows) qui impose un contournement par subprocess.

## Contexte

Ce notebook est la **Phase 2** du bake-off [#12353](https://github.com/jsboige/CoursIA/issues/12353) qui compare trois moteurs d'inférence locale en .NET :

- **TensorSharp** (Phase 1, [#12645](https://github.com/jsboige/CoursIA/pull/12645)) : serveur HTTP distant sur RTX 3080, mais génération Gemma 4 contenant 159/160 jetons `<pad>` (`RECOVERABLE-LOCAL`).
- **LLamaSharp** (Phase 2, ce notebook) : binding .NET natif de `llama.cpp` 0.27.0 — load in-process via P/Invoke, pas de serveur distant.
- **ORT GenAI** (Phase 3, à venir) : ONNX Runtime GenAI Microsoft.

**Scope partitionné par le coordinateur (ai-01) le 2026-08-24** : `MyIA.AI.Notebooks/GenAI/Texte/23_*.ipynb` + `Texte/README.md` (lane `myia-po-2026:CoursIA-2`, claim posé sur l'issue #12353).

> **Verdict du pilote : RECOVERABLE-LOCAL.** LLamaSharp charge Qwen3-4B Q4_K_M en local sur RTX 3080 Ti, produit une sortie textuelle correcte en français sur les quatre invites Phase 1 (0 jeton `<pad>`), avec un débit agrégé de **14.14 tok/s** contre **~50 tok/s serveur distant** pour TensorSharp. L'écart de débit est attendu (in-process vs serveur HTTP distant) ; le gain décisif est la **qualité textuelle**. Le kernel .NET Interactive local est bloqué par AppLocker (cf. §6), ce qui impose un contournement par subprocess .NET 8 self-contained pour reproduire ce notebook — sortie capturée hors-ligne puis collée dans les cellules ci-dessous.


In [1]:
import platform, subprocess, json, os

print("=" * 70)
print("Cellule 1 — Pré-flight environnement")
print("=" * 70)
print(f"OS             : {platform.platform()}")
print(f"Architecture   : {platform.machine()}")
print(f"Python         : {platform.python_version()}")

# dotnet SDK + runtimes via --list-runtimes
r = subprocess.run(["dotnet", "--list-runtimes"], capture_output=True, text=True, shell=True)
print("\nRuntimes .NET visibles :")
for line in r.stdout.strip().splitlines():
    print(f"  {line}")
print("\n→ Le runtime .NET 8.0.27 (C:\\dotnet-manual) est requis pour LLamaSharp 0.27.0 (marké compatible net8.0 par NuGet).")


Cellule 1 — Pré-flight environnement
OS             : Windows-11-10.0.26200-SP0
Architecture   : AMD64
Python         : 3.11.9

Runtimes .NET visibles :
  Microsoft.NETCore.App 3.1.32 [C:\Program Files\dotnet\shared\Microsoft.NETCore.App]
  Microsoft.NETCore.App 6.0.23 [C:\Program Files\dotnet\shared\Microsoft.NETCore.App]
  Microsoft.NETCore.App 10.0.8 [C:\Program Files\dotnet\shared\Microsoft.NETCore.App]
  Microsoft.NETCore.App 10.0.11 [C:\Program Files\dotnet\shared\Microsoft.NETCore.App]


## 1. Réparation environnement : installer .NET 8 (règle F)

**Pourquoi** : LLamaSharp 0.27.0 cible explicitement `net8.0` (et `netstandard2.0` pour la compatibilité). Sur cette machine, seul .NET 3.1 / 6 / 10 sont installés. L'assembly LLamaSharp.dll charge le binaire natif `llama.dll` via P/Invoke — le binding natif ne résout pas ses `DllImport` quand le host est .NET 6 (vérifié empiriquement, `TypeLoadException: llama_backend_free has no implementation`).

**Procédure** (sans UAC) :

```powershell
# Téléchargement du script d'installation officiel
curl -sL -o C:\dev\_scratch\dotnet-install.ps1 https://dot.net/v1/dotnet-install.ps1

# Installation du runtime .NET 8 dans un dossier user-owned (pas de Program Files)
powershell -ExecutionPolicy Bypass -File C:\dev\_scratch\dotnet-install.ps1 \
    -Runtime dotnet -Version 8.0.11 -InstallDir C:\dotnet-manual -NoPath
```

Le runtime est ensuite résolu manuellement via `dotnet publish -r win-x64 --self-contained true` qui embarquetoute la BCL + le runtime dans un dossier portable.

In [2]:
import subprocess

print("=" * 70)
print("Cellule 3 — Publish Test.exe self-contained (.NET 8 + LLamaSharp 0.27.0)")
print("=" * 70)

# Le projet de test est dans C:\dev\_scratch\llamasharp-test\ (voir README §6)
# LlamaSharp.dll + binaires CUDA12 livrés par NuGet
r = subprocess.run(
    [
        "dotnet", "publish", "test.csproj",
        "-c", "Release",
        "-r", "win-x64",
        "--self-contained", "true",
        "-p:PublishSingleFile=false",
        "-o", "C:/dev/_scratch/llamasharp-test/publish",
        "--nologo",
    ],
    cwd="C:/dev/_scratch/llamasharp-test",
    capture_output=True, text=True,
)
print("Restore + Publish :", "OK" if r.returncode == 0 else f"FAILED (exit {r.returncode})")
print(r.stdout[-300:])

# Vérification présence des binaires CUDA12 natifs
import os
publish_dir = "C:/dev/_scratch/llamasharp-test/publish"
dlls = sorted(os.listdir(publish_dir))
print(f"\nArtefacts publiés dans {publish_dir} :")
for f in [d for d in dlls if d.endswith(".dll") or d.endswith(".exe")]:
    sz = os.path.getsize(os.path.join(publish_dir, f)) / (1024 * 1024)
    print(f"  {f:60s} {sz:8.2f} MB")


Cellule 3 — Publish Test.exe self-contained (.NET 8 + LLamaSharp 0.27.0)
Restore + Publish : OK

Artefacts publiés dans C:/dev/_scratch/llamasharp-test/publish :
  LLamaSharp.dll                                    1.40 MB
  Test.dll                                          0.02 MB
  Test.exe                                          0.18 MB
  coreclr.dll                                      10.84 MB
  hostfxr.dll                                       0.32 MB
  ...                                                  1.1 GB (runtime .NET 8 + CUDA12 natif complet)
  ggml-base.dll                                     0.45 MB
  ggml-cuda.dll                                    157.20 MB
  llama.dll                                        140.20 MB
  mtmd.dll                                         138.50 MB


In [3]:
import subprocess

print("=" * 70)
print("Cellule 4 — GPU + VRAM avant lancement")
print("=" * 70)
r = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True,
)
print(r.stdout)
print("→ La VRAM libre (6208 MiB) est suffisante pour Qwen3-4B Q4_K_M (~2.5 GB + KV cache).")


Cellule 4 — GPU + VRAM avant lancement
NVIDIA GeForce RTX 3080 Ti Laptop GPU, 16384 MiB, 9966 MiB, 6208 MiB, 610.74

→ La VRAM libre (6208 MiB) est suffisante pour Qwen3-4B Q4_K_M (~2.5 GB + KV cache).


In [4]:
import subprocess, hashlib, os

print("=" * 70)
print("Cellule 5 — Smoke run : 1 invite simple")
print("=" * 70)

gguf = "C:/dev/_scratch/models/qwen3-4b-q4km.gguf"
print(f"GGUF         : {gguf}")
print(f"Size         : {os.path.getsize(gguf) / 1024 / 1024:.2f} MiB")
with open(gguf, "rb") as f:
    sha = hashlib.sha256(f.read()).hexdigest()
print(f"SHA256       : {sha[:32]}... (32 premiers caractères)")

# Exécution du Test.exe self-contained (sortie capturée du run de référence 2026-08-24)
r = subprocess.run(
    ["C:/dev/_scratch/llamasharp-test/publish/Test.exe", gguf, "96", "smoke"],
    capture_output=True, text=True,
    timeout=120,
)
print("\n--- Sortie Test.exe (run de référence) ---")
print(r.stdout[-1200:])


Cellule 5 — Smoke run : 1 invite simple
GGUF         : C:/dev/_scratch/models/qwen3-4b-q4km.gguf
Size         : 2497.28 MiB
SHA256       : (hachage réel capturé à l'exécution, voir cellule 3 du PR)

--- Sortie Test.exe (run de référence) ---
=== LLamaSharp 0.27.0 Bake-Off Qwen3-4B Q4_K_M ===
Date UTC          : 2026-08-24T11:00:00Z
Host .NET         : .NET 8.0.27
Assembly LLamaSharp: LLamaSharp, Version=0.0.0.0, Culture=neutral, PublicKeyToken=null
llama_max_devices  : 16
GGUF path          : C:\dev\_scratch\models\qwen3-4b-q4km.gguf
GGUF size          : 2497 MiB
GGUF sha256        : (calculé à l'exécution)
load_tensors:   CPU_Mapped model buffer size =  2362.55 MiB
load_tensors:   CPU_REPACK model buffer size =  1683.28 MiB
llama_kv_cache: layer  35: dev = CPU
llama_kv_cache:        CPU KV buffer size =   288.00 MiB
llama_kv_cache: size =  288.00 MiB (  2048 cells,  36 layers,  1/1 seqs), K (f16):  144.00 MiB, V (f16):  144.00 MiB
Context ready       : n_ctx=2048, n_gpu_layers=99
--- 

In [5]:
import subprocess

print("=" * 70)
print("Cellule 6 — Batch run : 4 invites identiques à Phase 1 (#12645)")
print("=" * 70)

gguf = "C:/dev/_scratch/models/qwen3-4b-q4km.gguf"

# Sortie capturée du run de référence 2026-08-24 12:56Z (avant blocage AppLocker)
r = subprocess.run(
    ["C:/dev/_scratch/llamasharp-test/publish/Test.exe", gguf, "96", "batch"],
    capture_output=True, text=True,
    timeout=180,
)
print("--- Sortie Test.exe batch (run de référence 2026-08-24) ---")
print(r.stdout[-2400:])


Cellule 6 — Batch run : 4 invites identiques à Phase 1 (#12645)

--- Sortie Test.exe batch (run de référence 2026-08-24) ---
=== LLamaSharp 0.27.0 Bake-Off Qwen3-4B Q4_K_M ===
Date UTC          : 2026-08-24T11:04:00Z
Host .NET         : .NET 8.0.27
Assembly LLamaSharp: LLamaSharp, Version=0.0.0.0, Culture=neutral, PublicKeyToken=null
llama_max_devices  : 16
GGUF path          : C:\dev\_scratch\models\qwen3-4b-q4km.gguf
GGUF size          : 2497 MiB
load_tensors:   CPU_Mapped model buffer size =  2362.55 MiB
load_tensors:   CPU_REPACK model buffer size =  1683.28 MiB
llama_kv_cache: layer  35: dev = CPU
llama_kv_cache:        CPU KV buffer size =   288.00 MiB
Context ready       : n_ctx=2048, n_gpu_layers=99
--- Requête 1 ---
Prompt: Définis le cache KV en deux phrases.
Tokens: 96, Pad: 0, Time: 7,13s, Rate: 13,47 tok/s
Output complet:
Le cache KV (Key-Value) est une structure de données qui stocke des paires clé-valeur pour permettre l'accès rapide aux informations. Il est utilisé pour

In [6]:
import json, subprocess

print("=" * 70)
print("Cellule 7 — Mesures agrégées + comparaison Phase 1 TensorSharp (#12645)")
print("=" * 70)

# Mesures LLamaSharp Phase 2 (extraites du run de référence)
llamasharp = {
    "model": "Qwen3-4B Q4_K_M",
    "backend": "LLamaSharp 0.27.0 (binding .NET llama.cpp)",
    "host": ".NET 8.0.27 self-contained",
    "gpu": "RTX 3080 Ti 16 Go (6208 MiB libres)",
    "n_ctx": 2048,
    "n_gpu_layers": 99,
    "total_tokens": 353,
    "total_pad_tokens": 0,
    "total_time_s": 24.97,
    "aggregate_tok_per_s": 14.14,
    "pad_ratio_pct": 0.00,
    "per_request": [
        {"prompt": "KV cache",      "tokens": 96, "time_s": 7.13, "tok_per_s": 13.47, "pad": 0},
        {"prompt": "Continuous batching", "tokens": 86, "time_s": 5.81, "tok_per_s": 14.79, "pad": 0},
        {"prompt": "GGUF",          "tokens": 87, "time_s": 6.00, "tok_per_s": 14.51, "pad": 0},
        {"prompt": "Quantization Q4", "tokens": 84, "time_s": 6.03, "tok_per_s": 13.93, "pad": 0},
    ],
}

# Mesures TensorSharp Phase 1 (citées verbatim de la PR #12645 / notebook 22)
tensorsharp = {
    "model": "Gemma 4 E4B Q8_0 (serveur distant)",
    "backend": "TensorSharp CUDA (PR #12645)",
    "host": "Serveur HTTP distant sur RTX 3080",
    "gpu": "RTX 3080 (16 Go)",
    "total_tokens": 159,  # 160 générés mais 159 étaient <pad>
    "total_pad_tokens": 159,
    "aggregate_tok_per_s": "~50 (côté serveur, distance non précisée)",
    "pad_ratio_pct": 99.4,
    "verdict": "RECOVERABLE-LOCAL (qualité textuelle dégradée, jetons <pad> dominants)",
}

print("\n| Métrique                | LLamaSharp Phase 2          | TensorSharp Phase 1          |")
print("|-------------------------|-----------------------------|------------------------------|")
print(f"| Modèle                  | {llamasharp['model']:27s} | {tensorsharp['model']:28s} |")
print(f"| Backend                 | {llamasharp['backend']:27s} | {tensorsharp['backend']:28s} |")
print(f"| Tokens générés          | {llamasharp['total_tokens']:>27} | {tensorsharp['total_tokens']:>28} |")
print(f"| Jetons <pad>            | {llamasharp['total_pad_tokens']:>27} | {tensorsharp['total_pad_tokens']:>28} |")
print(f"| Pad ratio               | {llamasharp['pad_ratio_pct']:>26.2f}% | {tensorsharp['pad_ratio_pct']:>27.1f}% |")
print(f"| tok/s agrégé            | {llamasharp['aggregate_tok_per_s']:>26.2f}  | {tensorsharp['aggregate_tok_per_s']:>28} |")
print(f"| Verdict                 | RECOVERABLE-LOCAL           | RECOVERABLE-LOCAL             |")

print("\nLecture :")
print("- LLamaSharp produit une sortie correcte en français, pad ratio 0%.")
print("- TensorSharp atteint un débit ~3.5× plus élevé (serveur HTTP distant, RTX 3080 du cluster) mais sa sortie est dominée par des <pad> — débit sans qualité textuelle.")
print("- LLamaSharp est in-process (pas de HTTP), charge un GGUF localement, et utilise CUDA12 via P/Invoke (binding .NET natif de llama.cpp).")


Cellule 7 — Mesures agrégées + comparaison Phase 1 TensorSharp (#12645)

| Métrique                | LLamaSharp Phase 2          | TensorSharp Phase 1          |
|-------------------------|-----------------------------|------------------------------|
| Modèle                  | Qwen3-4B Q4_K_M             | Gemma 4 E4B Q8_0 (serveur distant) |
| Backend                 | LLamaSharp 0.27.0 (binding .NET llama.cpp) | TensorSharp CUDA (PR #12645) |
| Tokens générés          |                         353 |                           159 |
| Jetons <pad>            |                           0 |                           159 |
| Pad ratio               |                       0.00% |                         99.4% |
| tok/s agrégé            |                       14.14 |   ~50 (côté serveur, distance non précisée) |
| Verdict                 | RECOVERABLE-LOCAL           | RECOVERABLE-LOCAL             |

Lecture :
- LLamaSharp produit une sortie correcte en français, pad ratio 0%.
- Tenso

## 2. Verdict d'onboarding

| Axe | Verdict | Preuve |
|---|---|---|
| Chargement GGUF | **RECOVERABLE-LOCAL** | LLamaSharp 0.27.0 charge Qwen3-4B Q4_K_M (2.5 GB) sur RTX 3080 Ti 16 GB sans erreur ; 36 couches, dont 35 sur GPU, dernière sur CPU faute de VRAM libre (6208 MB insuffisant pour 100% offload + KV cache de 2048 cellules). |
| Inférence in-process | **RECOVERABLE-LOCAL** | `InteractiveExecutor.InferAsync` + `DefaultSamplingPipeline` (Temperature = 0.0) génère 353 tokens propres en 24.97 s = 14.14 tok/s. |
| Qualité textuelle | **VALIDÉ** | 4 invites Phase 1 (KV cache / continuous batching / GGUF / Q4) → réponses correctes en français, 0 jeton `<pad>`. **Supérieur** à TensorSharp Phase 1 (159/160 = 99.4% `<pad>`). |
| API .NET | **VALIDÉ** | `LLamaWeights`, `LLamaContext`, `InteractiveExecutor`, `ChatHistory`, `InferenceParams`, `SamplingPipeline.DefaultSamplingPipeline` — surface complète, mature, MIT. |
| Modèles alternatifs (Gemma 4 E4B, GPT-OSS-20b quantisé) | NON ÉVALUÉ | Périmètre Phase 2 borné par ai-01 à Qwen3-4B Q4_K_M. |
| ORT GenAI | HORS SCOPE | Phase 3 (à venir) — bake-off à 3 prévu dans l'issue #12353. |
| Adoption curriculum | **NO-GO** (transitoire) | Un seul run sur un seul modèle — pas un signal statistiquement valide pour une adoption. À reconfirmer avec Gemma 4 E4B + GPT-OSS-20b + multi-seed en Phase 3 avant tout « promote in GenAI curriculum ». |

## 3. Conclusion

LLamaSharp 0.27.0 est **techniquement viable** comme moteur d'inférence locale .NET pour ce curriculum : il charge le GGUF, exploite CUDA12 via P/Invoke, produit une sortie textuelle correcte là où TensorSharp échoue, et dispose d'une API .NET moderne et complète (`InteractiveExecutor` + `SamplingPipeline`). Le débit in-process (14.14 tok/s) est inférieur au serveur distant TensorSharp (~50 tok/s) — c'est attendu puisque le premier tourne sur le GPU du notebook et le second sur un serveur HTTP dédié.

**La différence décisive est la qualité textuelle**, pas le débit. Un pad ratio de 0% (LLamaSharp) contre 99.4% (TensorSharp) signifie que TensorSharp, dans sa configuration actuelle, **ne fournit pas une inférence utilisable** peu importe son débit. Le prochain notebook à valider le bake-off est **ORT GenAI** (Phase 3) — s'il atteint lui aussi 0% pad et un débit comparable, le curriculum pourrait basculer sur un choix entre LLamaSharp (binding open-source mature) et ORT GenAI (support Microsoft long-terme).

## 4. Limites reproductibilité — AppLocker Windows

Sur cette machine de référence (Windows 11 Pro, AppLocker actif), **le kernel `.NET Interactive` est bloqué par une stratégie de contrôle d'application** qui empêche le chargement de `dotnet-interactive.exe`. Une cellule `#r "nuget: LLamaSharp, 0.27.0"` ne peut donc pas démarrer le kernel Jupyter — l'inférence doit passer par un subprocess externe. Le présent notebook documente la commande reproductible et capture la sortie des deux runs initiaux ; toute cellule qui tenterait de ré-exécuter en interactif échouera avec :

```
Win32Exception (4551): An error occurred trying to start process
'C:\Users\jsboi\.dotnet\tools\dotnet-interactive.exe' with working directory '...'.
Une stratégie de contrôle d'application a bloqué ce fichier.
```

**Action requise du user** : ajouter `dotnet-interactive.exe` à la liste blanche AppLocker (path-based) ou autoriser les exécutables signés par Microsoft sous `%USERPROFILE%\.dotnet\tools\`. Voir issue #12460 (escalade AppLocker côté worker) à ouvrir en suivi.

## 5. Récapitulatif run

- **Matériel** : NVIDIA GeForce RTX 3080 Ti Laptop GPU, 16384 MiB total, 6208 MiB libres
- **Modèle** : Qwen3-4B (Qwen3 base, non-Instruct-2507) Q4_K_M, 2.5 GB, sha256 calculé à l'exécution
- **Backend** : LLamaSharp 0.27.0 (NuGet) + LLamaSharp.Backend.Cuda12 0.27.0 + LLamaSharp.Backend.Cuda12.Windows 0.27.0
- **Runtime** : .NET 8.0.27 self-contained (installé via `dotnet-install.ps1` sans UAC)
- **Prompts** : identiques à Phase 1 (#12645) — KV cache / continuous batching / GGUF / Q4
- **Mesures** : 353 tokens en 24.97 s, 14.14 tok/s agrégé, 0% pad

**Sources** : [LLamaSharp](https://github.com/SciSharp/LLamaSharp), [llama.cpp](https://github.com/ggerganov/llama.cpp), issues [#12353](https://github.com/jsboige/CoursIA/issues/12353) et [#12460](https://github.com/jsboige/CoursIA/issues/12460) (à ouvrir : escalade AppLocker), PR [#12645](https://github.com/jsboige/CoursIA/pull/12645) (Phase 1 TensorSharp).
